# SCI1070 — Semaine 10 : PPO, ses trois boutons, et le compromis mesuré

| Volet | Ce que vous y faites |
| --- | --- |
| A | PPO écrit de bout en bout sur `Pendulum-v1` |
| B | régler \(\epsilon\), \(\lambda\) et les époques, avec graines et intervalles |
| C | une méthode hors politique sur la **même** tâche, et le compromis en deux figures |

Ce qui est noté cette semaine, c'est le **jalon 3**, qui réutilise votre code du
volet A. **Faites donc le laboratoire d'abord, sans exception.**

**Les récompenses de cette tâche sont négatives : votre courbe monte vers zéro.**

> **Quatre pièges.**
>
> 1. **Le détachement de l'ancienne log-probabilité.** Non détachée, votre rapport a
>    un gradient dans son dénominateur et vous n'optimisez pas ce que vous avez écrit.
> 2. **Le décalage d'un élément.** Les avantages sont plus courts d'une transition
>    que la trajectoire, et l'erreur **ne lève aucune exception**.
> 3. **L'échelle de l'action.** Multipliez par la demi-étendue, une fois.
> 4. **L'évaluation avec tirage.** L'évaluation se fait **sur la moyenne**.

In [ ]:
import importlib.util
import subprocess
import sys

for paquet, module in [("gymnasium[classic-control]>=1.0", "gymnasium"), ("torch", "torch")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", paquet], check=True)
if importlib.util.find_spec("sci1070") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "git+https://github.com/FaycalZM/drl-course-teluq.git#subdirectory=code"],
                   check=True)

import collections
import time

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sci1070.protocole import intervalle, se_recouvrent
from sci1070.courbes import tracer_bande
from sci1070.graines import fixer_graines

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GRAINES = [0, 1, 2]
print("périphérique :", device)

---
# Volet A : PPO de bout en bout

**Étape 1 — les deux réseaux**, **séparés**, avec **deux optimiseurs**. Écart-type
**global**, un simple `nn.Parameter`.

In [ ]:
class ActeurGaussien(nn.Module):
    def __init__(self, n_obs, n_actions, demi_etendue, n_cache=64):
        super().__init__()
        self.demi_etendue = float(demi_etendue)
        ...        # À COMPLÉTER : la moyenne par un réseau, l'écart-type par un nn.Parameter GLOBAL, et la demi-étendue au bon endroit

    def distribution(self, obs):
        moyenne, ecart_type = self(obs)
        return torch.distributions.Normal(moyenne, ecart_type)


class Critique(nn.Module):
    def __init__(self, n_obs, n_cache=64):
        super().__init__()
        ...        # À COMPLÉTER : un réseau séparé, pas un tronc partagé

**Étape 2 — la collecte.** **Mémorisez la log-densité au moment où vous jouez
l'action, et détachez-la.** C'est la seule donnée supplémentaire que PPO demande, et
**tout en dépend**.

In [ ]:
def collecter(env, acteur, n_transitions):
    """Rend (obs, actions, recompenses, termines, anciennes_logprob)."""
    ...        # À COMPLÉTER : Piège nº1 : .detach() sur les log-probabilités, ICI.

**Étape 3 — l'avantage généralisé**, en une passe arrière, puis la normalisation.

In [ ]:
def avantage_generalise(recompenses, valeurs, termines, gamma=0.99, lam=0.95):
    """Une passe arrière. Le tableau rendu est PLUS COURT d'un élément (piège nº2)."""
    ...        # À COMPLÉTER : la récurrence, et la longueur T-1

### Les trois vérifications arithmétiques

**La première est la plus rentable du cours.**

In [ ]:
anciennes_logprob = torch.tensor([-1.2, -0.7, -2.1])
nouvelles_logprob = anciennes_logprob.clone()          # la politique n'a pas bougé
ratio = torch.exp(nouvelles_logprob - anciennes_logprob)
print("rapport au premier minilot :", ratio)
assert torch.allclose(ratio, torch.ones_like(ratio)), "bogue dans l'ancienne log-densité"

a = torch.randn(512) * 3.7 + 1.4
a_norm = (a - a.mean()) / (a.std() + 1e-8)
print(f"après normalisation : moyenne {a_norm.mean():.2e}, écart-type {a_norm.std():.4f}")
assert abs(float(a_norm.mean())) < 1e-5 and abs(float(a_norm.std()) - 1) < 1e-3

In [ ]:
...        # À COMPLÉTER : 3. la longueur du tableau d'avantages vaut la longueur de la trajectoire moins un

**Étape 4 — la boucle.** \(\epsilon = 0{,}2\), \(\lambda = 0{,}95\), dix époques.

In [ ]:
def ppo(graine, eps=0.2, lam=0.95, epoques=10, n_transitions=2048,
        max_pas=120_000, gamma=0.99, taille_minilot=64):
    """Rend (observations, courbe_evaluation, diagnostic)."""
    ...        # À COMPLÉTER : la boucle en treize étapes de la section 10.2. `diagnostic` porte, par époque, la moyenne de ln r_t et la fraction d'échantillons écrêtés.

### Le diagnostic de la semaine

C'est votre **seul** diagnostic honnête : la perte de PPO ne vous dit rien, puisqu'elle
est l'opposé d'un objectif que l'écrêtage aplatit volontairement.

In [ ]:
observations, courbe_pilote, diagnostic = ppo(graine=0)
print("cible annoncée à partir de l'exécution pilote :", round(float(courbe_pilote.max()), 1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
...        # À COMPLÉTER : ln r_t à gauche, fraction écrêtée à droite, par époque
for ax in axes:
    ax.set_xlabel("numéro d'époque"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("à partir de l'époque où la fraction écrêtée plafonne, les époques "
      "suivantes ne déplacent plus la politique : c'est la réponse de l'étape 4.")

In [ ]:
courbes = [ppo(graine=g)[1] for g in GRAINES]

fig, ax = plt.subplots(figsize=(8, 4.5))
n = min(len(c) for c in courbes)
tracer_bande(ax, np.arange(n), [c[:n] for c in courbes], etiquette="PPO, référence")
ax.set_xlabel("évaluation"); ax.set_ylabel("retour, sans tirage, dix épisodes")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

---
# Volet B : régler les trois boutons

In [ ]:
CONFIGURATIONS = {
    "Référence":                dict(eps=0.2, lam=0.95, epoques=10),
    "Bande étroite":            dict(eps=0.1, lam=0.95, epoques=10),
    "Bande large":              dict(eps=0.4, lam=0.95, epoques=10),
    "Avantage à un pas":        dict(eps=0.2, lam=0.0,  epoques=10),
    "Avantage de la semaine 8": dict(eps=0.2, lam=1.0,  epoques=10),
    "Une seule époque":         dict(eps=0.2, lam=0.95, epoques=1),
    "Vingt époques":            dict(eps=0.2, lam=0.95, epoques=20),
}

resultats = {}
for nom, options in CONFIGURATIONS.items():
    scores = ...        # À COMPLÉTER : un score final par graine, LES MÊMES graines
    resultats[nom] = scores
    moyenne, bas, haut = intervalle(scores)
    print(f"{nom:26s} {moyenne:8.1f}   [{bas:.1f}, {haut:.1f}]   n={len(scores)}")

In [ ]:
ref = resultats["Référence"]
for nom in list(CONFIGURATIONS)[1:]:
    recouvre = se_recouvrent(ref, resultats[nom])
    verdict = "je ne mesure pas de différence" if recouvre else "séparé de la référence"
    print(f"{nom:26s} {verdict}")

**Les quatre questions.** La bande étroite ralentit-elle, et de combien ? La bande
large provoque-t-elle une instabilité, et **à quel moment** ? Que se passe-t-il à
\(\lambda = 1\) — et retrouvez-vous bien l'avantage de la semaine 8, c'est-à-dire le
retour Monte-Carlo moins la valeur, sans amorçage intermédiaire ? Et **au-delà de
combien d'époques votre score cesse-t-il de s'améliorer** ?

---
# Volet C : une méthode hors politique sur la même tâche

**Deux routes honnêtes, et je ne tranche pas pour vous.** La route courte prend
l'implémentation de bibliothèque et concentre votre travail sur la **mesure**. **Elle
est parfaitement acceptable, à une condition : dites-le.**

In [ ]:
# La solution courte, pour verifier votre implementation et pour le volet C.
# A executer APRES avoir ecrit la votre, jamais avant.
import time

from stable_baselines3 import PPO, SAC

RESULTATS = {}

for nom, Classe, pas in [("sur politique", PPO, 60_000),
                         ("hors politique", SAC, 20_000)]:
    debut = time.perf_counter()
    modele = Classe("MlpPolicy", "Pendulum-v1", seed=0, verbose=0)
    modele.learn(total_timesteps=pas)
    RESULTATS[nom] = (pas, time.perf_counter() - debut)

for nom, (pas, duree) in RESULTATS.items():
    print(f"{nom:>16} : {pas} observations en {duree:.0f} s")

### Étape 2 — les deux mesures, dans **deux figures différentes**

In [ ]:
...        # À COMPLÉTER : gauche : retour contre OBSERVATIONS ; droite : retour contre TEMPS D'HORLOGE
axes[0].set_xlabel("observations consommées")
axes[1].set_xlabel("temps d'horloge (s)")
axes[0].set_ylabel("retour, sans tirage")
axes[0].legend(); axes[1].legend()
plt.tight_layout(); plt.show()

### Étape 4 — la vérification d'honnêteté

**Combien de mises à jour de réseau chaque méthode effectue par observation
collectée.** Ce rapport est l'explication mécanique de tout ce que vos deux figures
montrent, et **il tient en un nombre**.

In [ ]:
...        # À COMPLÉTER : le nombre de mises à jour par observation, pour les deux

**Les trois questions.** Laquelle atteint votre cible avec le **moins
d'observations** ? Laquelle en **le moins de temps** ? Et, la question qui ferme le
module : sur **votre** matériel, laquelle choisiriez-vous, et **qu'est-ce qui est
rare chez vous** ?

> **Mon message.** La perte de PPO ne vous dit **rien** sur la santé de votre
> entraînement. Votre seul diagnostic honnête est le couple que le volet A demande de
> tracer. Et **une comparaison entre deux méthodes dont l'une est mal réglée ne
> mesure que le réglage**.